In [ ]:
import numpy as np
import re
import matplotlib.pyplot as plt

In [ ]:

# 1. Load text from file
def load_text(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        text = f.read()
    return text

In [ ]:

# 2. Segment text into chunks (e.g., every 1000 words)
def segment_text(text, words_per_segment=1000):
    # Use regex to extract words (all lowercase)
    words = re.findall(r'\w+', text.lower())
    segments = []
    for i in range(0, len(words), words_per_segment):
        segment = words[i:i+words_per_segment]
        segments.append(segment)
    return segments

In [ ]:

# 3. Define a domain-specific lexicon for conspiratorial/semiotic keywords
lexicon = {
    "conspiracy": 0.8,
    "secret": 0.7,
    "cipher": 0.9,
    "mystery": 0.6,
    "occult": 0.85,
    "symbol": 0.5,
    "code": 0.7,
    "ritual": 0.6,
    "hidden": 0.65,
    "enigmatic": 0.75,
    # Additional keywords can be added here
}

In [ ]:

# 4. Compute thematic metrics for each segment:
#    - Density: sum of absolute intensities of keywords.
#    - Pressure: sum of intensities for keywords above a threshold.
#    - Viscosity: standard deviation of the intensities within a segment.
def compute_metrics(segments, lexicon, pressure_threshold=0.7):
    density = []
    pressure = []
    viscosities = []
    for segment in segments:
        intensities = []
        dens = 0
        pres = 0
        for word in segment:
            if word in lexicon:
                intensity = lexicon[word]
                intensities.append(intensity)
                dens += abs(intensity)
                if intensity >= pressure_threshold:
                    pres += intensity
        density.append(dens)
        if intensities:
            viscosities.append(np.std(intensities))
        else:
            viscosities.append(0)
        pressure.append(pres)
    return np.array(density), np.array(pressure), np.array(viscosities)

In [ ]:

# 5. Compute "velocity" as the change in density between segments.
def compute_velocity(density, delta_t=1):
    velocity = np.diff(density) / delta_t
    velocity = np.insert(velocity, 0, 0)  # prepend to match array length
    return velocity

In [ ]:

# 6. Compute an "external force" based on the average intensity of keywords in each segment.
def compute_external_force(segments, lexicon):
    external_force = []
    for segment in segments:
        values = []
        for word in segment:
            if word in lexicon:
                values.append(lexicon[word])
        if values:
            ext = np.mean(values)
        else:
            ext = 0
        external_force.append(ext)
    return np.array(external_force)

In [ ]:

# 7. Simulate thematic evolution (e) using a discrete Navier–Stokes-inspired update.
def simulate_theme_evolution(density, pressure, viscosity, external_force, delta_t=1, delta_x=1):
    n = len(density)
    # Initialize e: the thematic "state" for each segment.
    e = np.zeros(n)
    e[0] = density[0]  # for instance, starting with the density of the first segment
    if n > 1:
        e[1] = e[0]  # simple initialization for the second segment
    for i in range(1, n-1):
        # Compute gradient of pressure: (p[i+1] - p[i]) / delta_x
        grad_p = (pressure[i+1] - pressure[i]) / delta_x
        # Laplacian of e: (e[i+1] - 2*e[i] + e[i-1]) / (delta_x^2)
        lap_e = (e[i+1] - 2 * e[i] + e[i-1]) / (delta_x ** 2)
        # Compute convective term: use change in density as a proxy for "flow"
        convective = e[i] * ((density[i+1] - density[i]) / delta_x)
        # Avoid division by zero: if density[i] is zero, factor becomes 0.
        factor = 1 / density[i] if density[i] != 0 else 0
        # Update rule for e (thematic evolution)
        e[i+1] = e[i] + delta_t * ( - factor * grad_p + viscosity[i] * lap_e + external_force[i] - convective )
    return e

In [ ]:

# 8. Main function for Pipeline A
def pipeline_a(filename):
    # Load and preprocess the text
    text = load_text(filename)
    segments = segment_text(text, words_per_segment=1000)

    # Compute the fluid-inspired metrics
    density, pressure, viscosity = compute_metrics(segments, lexicon)
    velocity = compute_velocity(density)
    external_force = compute_external_force(segments, lexicon)

    # Simulate the thematic evolution using the modified Navier-Stokes equation
    theme_evolution = simulate_theme_evolution(density, pressure, viscosity, external_force)

    # Visualization of the computed metrics and simulation output
    segments_range = np.arange(len(density))

    plt.figure(figsize=(12, 10))

    plt.subplot(3, 1, 1)
    plt.plot(segments_range, density, label='Density', marker='o')
    plt.plot(segments_range, pressure, label='Pressure', marker='x')
    plt.ylabel('Metric Value')
    plt.title('Fluid Dynamics-Inspired Thematic Metrics')
    plt.legend()

    plt.subplot(3, 1, 2)
    plt.plot(segments_range, velocity, label='Velocity', color='orange', marker='d')
    plt.ylabel('Velocity')
    plt.title('Rate of Change of Density')
    plt.legend()

    plt.subplot(3, 1, 3)
    plt.plot(segments_range, theme_evolution, label='Theme Evolution (e)', color='green', marker='s')
    plt.xlabel('Segment Index')
    plt.ylabel('e (Thematic State)')
    plt.title('Simulated Thematic Evolution')
    plt.legend()

    plt.tight_layout()
    plt.show()

    # Return all computed metrics for further analysis
    return {
        'density': density,
        'pressure': pressure,
        'viscosity': viscosity,
        'velocity': velocity,
        'external_force': external_force,
        'theme_evolution': theme_evolution
    }

In [ ]:

# 9. Execute the pipeline if run as a script
if __name__ == "__main__":
    # Replace 'the_foucault_pendulum.txt' with the path to your text file
    results = pipeline_a('the_foucault_pendulum.txt')
